# Missing data cho Data Analyst

Khi Analyst mở bảng và thấy `NaN`, việc đầu tiên **không** phải là `.fillna()` ngay.

Quy trình đúng:

1. **Đếm** — cột nào thiếu, thiếu bao nhiêu %.
2. **Hiểu vì sao thiếu** — lỗi hệ thống, người không điền, hay giá trị nhạy cảm.
3. **Chọn cách xử lý** — xoá, điền, hay giữ “không rõ”.
4. **Ghi chú** — cách điền làm thay đổi phân phối / metric thế nào.

MCAR / MAR / MNAR là cách gọi 3 *lý do thiếu*. Chúng giúp chọn bước 3, không phải tên hàm pandas.

## 1. Analyst nhìn missing thế nào?

Mỗi dòng là một quan sát (khách, đơn hàng, phiếu khảo sát). Mỗi cột là một biến.

- **Biến số** (`int` / `float`): tuổi, doanh thu, số lần mua → có mean, median.
- **Biến phân loại** (`object` / `category`): giới tính, kênh bán, loại thẻ → chỉ đếm tần suất, **không có mean**.

`NaN` = giá trị không quan sát được. Nếu bỏ qua, dashboard (tỷ lệ, trung bình, hệ số tương quan) có thể **sai** vì chỉ tính trên phần còn lại — phần còn lại có thể không đại diện cho toàn bộ.

Ví dụ: 20% khách không khai thu nhập. Nếu đúng người thu nhập cao không khai, “thu nhập trung bình khách hàng” tính trên 80% còn lại sẽ **thấp hơn thực tế**.

## 2. Ba cơ chế thiếu 

Câu hỏi **việc ô bị trống có liên quan giá trị thật của ô đó không?**

### MCAR — thiếu hoàn toàn ngẫu nhiên

Thiếu **không liên quan** bất kỳ biến nào (kể cả chính ô đó).

- File export lỗi vài dòng bất kỳ; nhân viên quên tick ngẫu nhiên.
- Phần còn lại **vẫn đại diện** cho tổng thể.
- Hệ quả: xoá dòng hoặc điền mean/mode **ít bias**. Chỉ **mất cỡ mẫu** (ước lượng kém chắc hơn).

### MAR — thiếu phụ thuộc biến đã có

Thiếu **giải thích được bằng cột khác**, không cần biết đúng giá trị đang trống.

- Khách **online** bỏ trống “cửa hàng gần nhất” nhiều hơn khách **offline** (đã có cột kênh).
- Nam trả lời khảo sát thu nhập ít hơn nữ (đã có cột giới tính).
- Phần còn lại **không** đại diện nếu không **khống chế theo nhóm**: trung bình thu nhập gộp sẽ lệch về nhóm chịu điền.
- Hệ quả: điền **theo nhóm** (mean/mode trong từng kênh, từng giới tính), không điền một số cho cả cột.

Tên *Missing At Random* dễ hiểu nhầm: **không** phải “ngẫu nhiên”. Nghĩa là: *đã biết các cột khác thì việc thiếu không còn phụ thuộc giá trị đang thiếu*.

### MNAR — thiếu phụ thuộc đúng giá trị đang giấu

Người ta **không điền vì giá trị đó**.

- Không khai lương vì lương rất cao (hoặc rất thấp).
- Không chọn “đã từng khiếu nại” vì không muốn lộ.
- Ô trống **là tín hiệu**. Điền mode/mean = giả vờ họ giống số đông → **bias mạnh**, KPI sai.
- Hệ quả: tạo nhãn `Unknown` / cột `is_missing`; không dùng mode để gán giá trị thay thế.

Analyst **không chứng minh được** MAR vs MNAR chỉ bằng file. Phải dựa vào quy trình thu thập (form nào bắt buộc, ai hay bỏ qua). MCAR đôi khi kiểm tra được sơ (thiếu có liên quan cột khác không).

## 3. Vì sao biến phân loại điền **mode**?

Đây là **single imputation**: thay mọi `NaN` bằng **một** giá trị.

| Loại biến | Giá trị điền phổ biến | Lý do Analyst |
|---|---|---|
| Số, phân phối đều | **Mean** | Giữ mức trung bình cột |
| Số, có outlier / lệch | **Median** | Không bị một giá trị cực đoan kéo |
| Phân loại (nominal) | **Mode** | Mean không tồn tại; mode = lớp có tần suất cao nhất |
| Thứ bậc (ordinal) | Median hoặc mode | Có thứ tự thì median còn nghĩa |

Mode = “nếu bắt buộc chọn 1 nhãn cho ô trống, chọn nhãn **hay gặp nhất** trong dữ liệu đã thấy.”

Chỉ hợp khi:

- Thiếu **ít** và gần **MCAR**, hoặc
- Chỉ cần bảng chạy được cho EDA sơ bộ, chấp nhận bias nhỏ.

Analyst phải biết mode làm gì với báo cáo:

- **Bias về lớp đa số** — 80% `Standard`, 20% `Premium`; mọi NaN thành `Standard` → tỷ lệ Premium **giảm** trên giấy.
- **Phương sai giảm** — cột trông “sạch, chắc” hơn thật; model/dashboard đánh giá quá tự tin.
- **Mất thông tin missing** — nếu MNAR, mất luôn tín hiệu “không khai”.

Vì vậy mode là **mặc định nhập môn**, không phải best practice cho mọi dashboard.

## 4. Menu xử lý 

| Mục đích | Nên làm |
|---|---|
| Đếm, vẽ phân phối mô tả | Ghi rõ “tính trên N dòng có dữ liệu”; hoặc `Unknown` để thấy tỷ lệ trống |
| So sánh nhóm (nam/nữ, kênh) | Điền / phân tích **trong từng nhóm** (MAR) |
| Biến dùng làm **target** (cần dự đoán) | **Không điền**; xoá dòng thiếu nhãn |
| Feature cho model | `Unknown` + `is_missing`, hoặc imputation theo nhóm; cân nhắc MICE nếu thiếu nhiều |
| Thiếu > ~50% một cột, ít giá trị nghiệp vụ | Cân nhắc **bỏ cột** |

Luôn **log**: cột nào, % missing, cơ chế giả định, phương pháp, trước/sau (value_counts hoặc mean).

## 5. Thực hành — đọc từng cột


12 khách, 2 kênh (`web` / `app`), 3 gói (`basic` / `plus` / `vip`).

Ý tưởng pandas: `s.where(điều_kiện, np.nan)`  
→ **giữ** giá trị nếu điều kiện đúng; **thành NaN** nếu sai.

In [25]:
import pandas as pd

df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "channel": ["web"] * 6 + ["app"] * 6,
    "plan": [  
        "basic", "basic", "plus", "vip", "basic", "plus",
        "basic", "plus", "vip", "basic", "plus", "vip",
    ],
})

# MCAR: xoá 2 dòng bất kỳ (id 2 và 9) — không liên quan gói hay kênh
df["plan_MCAR"] = df["plan"].where(~df["id"].isin([2, 9]))

# MAR: khách app hay bỏ trống (id 8, 10, 12) — thiếu vì channel, không vì plan
df["plan_MAR"] = df["plan"].where(~((df["channel"] == "app") & df["id"].isin([8, 10, 12])))

# MNAR: gói vip không khai (mọi dòng vip) — thiếu vì đúng giá trị plan
df["plan_MNAR"] = df["plan"].where(df["plan"] != "vip")

df

,id,channel,plan,plan_MCAR,plan_MAR,plan_MNAR
0,1,web,basic,basic,basic,basic
1,2,web,basic,NaN,basic,basic
2,3,web,plus,plus,plus,plus
3,4,web,vip,vip,vip,NaN
4,5,web,basic,basic,basic,basic
5,6,web,plus,plus,plus,plus
6,7,app,basic,basic,basic,basic
7,8,app,plus,plus,NaN,plus
8,9,app,vip,NaN,vip,NaN
9,10,app,basic,basic,NaN,basic


## 6. Xử lý từng bước: `mode` rồi `fillna`

Analyst **không sửa sự thật**. Chỉ làm việc với cột đang có `NaN` (ví dụ `plan_MCAR`).

Ba thao tác pandas:

1. `.value_counts()` — đếm mỗi nhãn (và `NaN` nếu `dropna=False`).
2. `.mode()` — nhãn **nhiều nhất** trong phần **đã điền** (bỏ `NaN` khi tính).
3. `.fillna(mode)` — mọi ô `NaN` **đổi thành đúng nhãn đó**. Ô đã có giá trị **giữ nguyên**.

Không phải thuật toán phức tạp: trống → ghi giống số đông.

In [26]:
def xu_ly_mode(ten_cot):
    """In từng bước: ai đang trống → mode là gì → ô trống thành gì."""
    truoc = df[ten_cot]
    print("=" * 60)
    print("CỘT:", ten_cot)
    print("=" * 60)

    print("\n[1] Những khách đang TRỐNG (NaN):")
    trong = df.loc[truoc.isna(), ["id", "channel", "plan", ten_cot]]
    print(trong.to_string(index=False) if len(trong) else "  (không có)")

    print("\n[2] Đếm nhãn trên phần ĐÃ ĐIỀN (bỏ NaN) — mode = nhãn thắng:")
    print(truoc.value_counts(dropna=True))
    mode = truoc.mode(dropna=True)[0]
    print("  → mode =", mode, "(dùng để điền chỗ trống)")

    sau = truoc.fillna(mode)
    df[ten_cot + "_sau"] = sau

    print("\n[3] Chỉ các dòng từng trống — trước vs sau fillna:")
    so_sanh = df.loc[truoc.isna(), ["id", "channel", "plan"]].copy()
    so_sanh["truoc"] = truoc[truoc.isna()].values
    so_sanh["sau"] = sau[truoc.isna()].values
    print(so_sanh.to_string(index=False))

    print("\n[4] Đếm cả 12 khách:")
    print("  sự thật     :", df["plan"].value_counts().to_dict())
    print("  trước điền  :", truoc.value_counts(dropna=False).to_dict())
    print("  sau điền    :", sau.value_counts().to_dict())
    print()

xu_ly_mode("plan_MCAR")
xu_ly_mode("plan_MAR")
xu_ly_mode("plan_MNAR")

print("MNAR — gán Unknown thay vì mode:")
print(df["plan_MNAR"].fillna("Unknown").value_counts(dropna=False).to_dict())

CỘT: plan_MCAR

[1] Những khách đang TRỐNG (NaN):
 id channel  plan plan_MCAR
  2     web basic       NaN
  9     app   vip       NaN

[2] Đếm nhãn trên phần ĐÃ ĐIỀN (bỏ NaN) — mode = nhãn thắng:
plan_MCAR
basic    4
plus     4
vip      2
Name: count, dtype: int64
  → mode = basic (dùng để điền chỗ trống)

[3] Chỉ các dòng từng trống — trước vs sau fillna:
 id channel  plan truoc   sau
  2     web basic   NaN basic
  9     app   vip   NaN basic

[4] Đếm cả 12 khách:
  sự thật     : {'basic': 5, 'plus': 4, 'vip': 3}
  trước điền  : {'basic': 4, 'plus': 4, nan: 2, 'vip': 2}
  sau điền    : {'basic': 6, 'plus': 4, 'vip': 2}

CỘT: plan_MAR

[1] Những khách đang TRỐNG (NaN):
 id channel  plan plan_MAR
  8     app  plus      NaN
 10     app basic      NaN
 12     app   vip      NaN

[2] Đếm nhãn trên phần ĐÃ ĐIỀN (bỏ NaN) — mode = nhãn thắng:
plan_MAR
basic    4
plus     3
vip      2
Name: count, dtype: int64
  → mode = basic (dùng để điền chỗ trống)

[3] Chỉ các dòng từng trống — trước vs s

Cách đọc kết quả (cùng một phương pháp imputation, ba cơ chế missing khác nhau).

Trong **mô phỏng** ta biết **giá trị gốc** (`plan`). Thực tế Analyst **không** có cột này — chỉ thấy cột có `NaN`. So sánh gốc vs sau imputation để đo **sai lệch**.

**MCAR — missing ở id 2 và 9**

Giá trị gốc: id 2 = `basic`, id 9 = `vip`.  
`fillna(mode)` gán **cả hai** thành `basic` (mode của phần đã quan sát).

- Id 2: giá trị gán **trùng** giá trị gốc.
- Id 9: giá trị gán **khác** giá trị gốc (vip bị thay bằng basic).

Vì missing không tập trung một nhóm, **tỷ trọng** các gói sau imputation **gần** phân phối gốc. Vẫn có sai lệch nhỏ: mọi `NaN` đều thành lớp đa số nên tỷ trọng `basic` tăng.

**MAR — missing ở id 8, 10, 12 (cùng `channel = app`)**

Cùng imputation: ba `NaN` thành `basic`.  
Missing **tương quan với biến đã quan sát** (`channel`), không rải đều. Tỷ trọng `basic` trên nhóm app **bị thổi** — đây là **bias** do single imputation toàn cục, không phải vì app thật sự dùng nhiều gói basic.

**MNAR — missing ở id 4, 9, 12 (cùng giá trị gốc `vip`)**

Ba bản ghi vip không còn trên dữ liệu đã quan sát. Imputation bằng mode → **tỷ trọng vip = 0**.  
KPI/tỷ trọng gói cao cấp **bị underestimate**. Phù hợp hơn: giữ missing như một mức `Unknown` (không gán mode).

**Phương pháp:** single imputation bằng **mode** = thay mọi `NaN` bằng nhãn có tần suất cao nhất trên dữ liệu đã quan sát. Độ lệch của metric phụ thuộc **cơ chế missing**, không phụ thuộc tên hàm pandas.

## 7. Làm sao biết là MCAR, MAR hay MNAR?


### Bước A — Hỏi người thu thập (quan trọng nhất)

| Câu hỏi | Thiên về |
|---|---|
| Lỗi export / form sập / nhân viên quên vài dòng bất kỳ? | **MCAR** |
| Câu hỏi *không bắt buộc*, hoặc *skip logic* (app thì không hỏi cửa hàng)? | **MAR** |
| Câu *nhạy cảm* (lương, bệnh, khiếu nại, “không muốn nói”)? | **MNAR** |

### Bước B — Soi trên dữ liệu đã có (phân biệt MCAR vs MAR)

Tạo cột `is_missing` = ô đó trống hay không. Xem `is_missing` có **dính** cột khác không.

- Thiếu **đều** mọi kênh, giới tính, tháng →  **MCAR**.
- Thiếu **dồn** một nhóm (app nhiều hơn web) →  **MAR**.

Ô dưới: đếm % trống theo `channel` trên 3 cột giả lập.

### Bước C — MNAR thì suy từ nghiệp vụ

File **không** nói được “vip hay giấu”. Chỉ biết: nếu lý do bỏ trống *là đúng cái đang hỏi* → MNAR → đừng `fillna(mode)`.

Thực tế Analyst:

- Ít khi tin **MCAR** (trừ lỗi kỹ thuật, thiếu rất ít).
- **MAR** = giả định làm việc được (điền / phân tích theo nhóm).
- **MNAR** khi câu hỏi nhạy cảm hoặc có lựa chọn “không khai”.


In [27]:
print("% trống theo kênh — nếu hai kênh gần bằng nhau → nghi MCAR")
print("                         nếu lệch hẳn một kênh → không MCAR (kiểu MAR)\n")

for col in ["plan_MCAR", "plan_MAR", "plan_MNAR"]:
    pct = df.groupby("channel")[col].apply(lambda s: s.isna().mean())
    print(col, "→ % NaN theo channel:")
    print((pct * 100).round(0).astype(int).to_string())
    print()

% trống theo kênh — nếu hai kênh gần bằng nhau → nghi MCAR
                         nếu lệch hẳn một kênh → không MCAR (kiểu MAR)

plan_MCAR → % NaN theo channel:
channel
app    17
web    17

plan_MAR → % NaN theo channel:
channel
app    50
web     0

plan_MNAR → % NaN theo channel:
channel
app    33
web    17



Kỳ vọng khi chạy ô trên (12 khách):

- `plan_MCAR`: web và app đều có trống → không dính một kênh.
- `plan_MAR`: **app trống nhiều, web 0%** → thiếu phụ thuộc `channel` → MAR, không MCAR.
- `plan_MNAR`: trống rải hai kênh (luật là *vip*). Bảng **không đủ** để gọi MNAR; phải biết “vip hay không khai”.

| Giả định | Làm gì |
|---|---|
| MCAR, thiếu ít | Mode (chữ) / mean (số), hoặc xoá dòng |
| MAR | Mode/mean **trong từng nhóm** |
| MNAR | `Unknown` / `is_missing`; không gán mode cho cả cột |

## 8. Checklist trước khi fillna

1. Cột này là **số hay phân loại**? Target hay feature?
2. Thiếu bao nhiêu %? **Nhóm nào** thiếu nhiều hơn?
3. Hỏi nguồn: lỗi kỹ thuật / skip form / câu nhạy cảm?
4. Chọn: drop / mean-median / mode / mode theo nhóm / `Unknown`.
5. So sánh metric trước–sau rồi mới đưa vào báo cáo.

Chọn MCAR/MAR/MNAR bằng *cách thu thập + xem ai đang trống*, không bằng tên hàm pandas.